# Reddit Sentiment vs. Stock Market — Correlation Analysis
**Percival Mahwaya | Social Media Trend Analysis**
**Team Project Semester 4 — Integration with Shamil Mammadrzayev (Stock Market)**

This notebook analyses the relationship between Reddit engagement/sentiment data (collected via PRAW) and stock market metrics (collected via yfinance by Shamil).

**Companies:** Apple (AAPL), Tesla (TSLA), Amazon (AMZN)
**Period:** March 2025 – March 2026
**Reddit data:** 277 posts → aggregated to 106 weekly records
**Stock data:** 840 daily rows → aggregated to 168 weekly records

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
REDDIT_PATH = r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\reddit_weekly_sentiment.csv"
STOCK_BASE  = r"C:\Users\Percival Mahwaya\Desktop\MSc DS\Sem 4\SIPDiDS Team Project\stock-market" + "\\"

# ── Load ──────────────────────────────────────────────────────────────────────
reddit_df = pd.read_csv(REDDIT_PATH)
corr_df   = pd.read_csv(STOCK_BASE + "correlation_dataset.csv")
vol_df    = pd.read_csv(STOCK_BASE + "weekly_volatility.csv")
stock_df  = pd.read_csv(STOCK_BASE + "stock_data_final.csv", parse_dates=["date"])

print("Reddit weekly data:", reddit_df.shape)
print("Correlation dataset:", corr_df.shape)
print("Weekly volatility:",   vol_df.shape)
print("Daily stock data:",    stock_df.shape)

## 2. Explore the Merged Dataset

The `correlation_dataset.csv` (built by Shamil) already merges Reddit weekly data with stock weekly data.
We use it as the primary analysis table.

In [ ]:
corr_df.head(5)

In [ ]:
print("Total weeks (all tickers):", len(corr_df))
print("Weeks WITH Reddit data:   ", corr_df["avg_score"].notna().sum())
print("Weeks WITHOUT Reddit data:", corr_df["avg_score"].isna().sum())
print()
print(corr_df.groupby("ticker")["avg_score"].apply(lambda x: f"{x.notna().sum()} reddit / {len(x)} total"))

In [ ]:
# Filter to only weeks that have Reddit data
reddit_weeks = corr_df[corr_df["avg_score"].notna()].copy()
print("Working dataset shape:", reddit_weeks.shape)
reddit_weeks.describe().round(3)

## 3. Pearson & Spearman Correlation

We test the correlation between:
- **Reddit features:** `post_count`, `avg_score`, `avg_upvote_ratio`, `avg_comments`, `avg_sentiment`
- **Stock metrics:** `weekly_price_change_pct`, `next_week_price_change_pct`, `weekly_volatility`, `avg_daily_return`

**Pearson** measures linear correlation.
**Spearman** measures rank-order correlation (more robust to outliers).

In [ ]:
reddit_features = ["post_count", "avg_score", "avg_upvote_ratio", "avg_comments", "avg_sentiment"]
stock_metrics   = ["weekly_price_change_pct", "next_week_price_change_pct", "weekly_volatility", "avg_daily_return"]

# ── Pearson ───────────────────────────────────────────────────────────────────
pearson_matrix  = pd.DataFrame(index=reddit_features, columns=stock_metrics, dtype=float)
pearson_pvalues = pd.DataFrame(index=reddit_features, columns=stock_metrics, dtype=float)

for rf in reddit_features:
    for sm in stock_metrics:
        valid = reddit_weeks[[rf, sm]].dropna()
        if len(valid) > 4:
            r, p = stats.pearsonr(valid[rf], valid[sm])
            pearson_matrix.loc[rf, sm]  = round(r, 3)
            pearson_pvalues.loc[rf, sm] = round(p, 4)

print("=== Pearson Correlation Matrix ===")
print(pearson_matrix.to_string())

In [ ]:
# ── Spearman ──────────────────────────────────────────────────────────────────
spearman_matrix  = pd.DataFrame(index=reddit_features, columns=stock_metrics, dtype=float)
spearman_pvalues = pd.DataFrame(index=reddit_features, columns=stock_metrics, dtype=float)

for rf in reddit_features:
    for sm in stock_metrics:
        valid = reddit_weeks[[rf, sm]].dropna()
        if len(valid) > 4:
            r, p = stats.spearmanr(valid[rf], valid[sm])
            spearman_matrix.loc[rf, sm]  = round(r, 3)
            spearman_pvalues.loc[rf, sm] = round(p, 4)

print("=== Spearman Correlation Matrix ===")
print(spearman_matrix.to_string())

## 4. Correlation Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, matrix, title in zip(axes,
                               [pearson_matrix.astype(float), spearman_matrix.astype(float)],
                               ["Pearson Correlation", "Spearman Correlation"]):
    sns.heatmap(
        matrix, ax=ax, annot=True, fmt=".3f",
        cmap="RdYlGn", vmin=-1, vmax=1,
        linewidths=0.5, mask=matrix.isna(),
        cbar_kws={"label": "Correlation (r)"}
    )
    ax.set_title(f"{title}\nReddit Features vs Stock Metrics", fontsize=13, fontweight="bold")
    ax.set_xlabel("Stock Metrics", fontsize=11)
    ax.set_ylabel("Reddit Features", fontsize=11)
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Reddit Sentiment vs Stock Market — Correlation Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\correlation_heatmap.png",
            bbox_inches="tight")
plt.show()
print("Saved: correlation_heatmap.png")

## 5. Per-Ticker Correlation

Does the Reddit–stock relationship differ by company?

In [ ]:
tickers      = ["AAPL", "TSLA", "AMZN"]
ticker_names = {"AAPL": "Apple", "TSLA": "Tesla", "AMZN": "Amazon"}
colors       = {"AAPL": "#1f77b4", "TSLA": "#d62728", "AMZN": "#ff7f0e"}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, ticker in zip(axes, tickers):
    sub = reddit_weeks[reddit_weeks["ticker"] == ticker]
    matrix = pd.DataFrame(index=reddit_features, columns=stock_metrics, dtype=float)
    for rf in reddit_features:
        for sm in stock_metrics:
            valid = sub[[rf, sm]].dropna()
            if len(valid) > 3:
                r, _ = stats.pearsonr(valid[rf], valid[sm])
                matrix.loc[rf, sm] = round(r, 3)
    sns.heatmap(matrix.astype(float), ax=ax, annot=True, fmt=".3f",
                cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.5,
                cbar_kws={"label": "r"})
    ax.set_title(f"{ticker_names[ticker]} ({ticker})", fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Pearson Correlation per Company", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\correlation_per_ticker.png",
            bbox_inches="tight")
plt.show()

## 6. Scatter Plots — Key Relationships

Visualise the most interesting pairs: sentiment vs price change, and Reddit score vs volatility.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

pairs = [
    ("avg_sentiment",   "weekly_price_change_pct",    "Sentiment vs Same-Week Price Change (%)"),
    ("avg_sentiment",   "next_week_price_change_pct", "Sentiment vs Next-Week Price Change (%)"),
    ("avg_score",       "weekly_volatility",           "Reddit Score vs Weekly Volatility"),
    ("post_count",      "weekly_volatility",           "Post Count vs Weekly Volatility"),
    ("avg_comments",    "weekly_price_change_pct",    "Avg Comments vs Price Change (%)"),
    ("avg_upvote_ratio","weekly_price_change_pct",    "Upvote Ratio vs Price Change (%)"),
]

for ax, (x_col, y_col, title) in zip(axes.flat, pairs):
    for ticker in tickers:
        sub = reddit_weeks[reddit_weeks["ticker"] == ticker][[x_col, y_col]].dropna()
        ax.scatter(sub[x_col], sub[y_col], label=ticker,
                   color=colors[ticker], alpha=0.7, edgecolors="white", s=70)
        if len(sub) > 3:
            m, b, r, p, _ = stats.linregress(sub[x_col], sub[y_col])
            x_range = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
            ax.plot(x_range, m * x_range + b, color=colors[ticker], linewidth=1.2, linestyle="--")
    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(x_col.replace("_", " ").title(), fontsize=9)
    ax.set_ylabel(y_col.replace("_", " ").title(), fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle("Scatter Plots: Reddit Features vs Stock Metrics", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\scatter_plots.png",
            bbox_inches="tight")
plt.show()

## 7. Time-Series Overlay — Reddit Sentiment vs Stock Price

Plot Reddit weekly sentiment alongside stock closing price to see temporal alignment.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

for ax, ticker in zip(axes, tickers):
    stock_sub = stock_df[stock_df["ticker"] == ticker].copy()
    stock_sub["year_week"] = stock_sub["date"].dt.strftime("%G-W%V")
    weekly_close = stock_sub.groupby("year_week")["close"].last().reset_index()

    r_sub = reddit_weeks[reddit_weeks["ticker"] == ticker][["year_week", "avg_sentiment"]].dropna()
    merged = weekly_close.merge(r_sub, on="year_week", how="left")
    merged["x"] = range(len(merged))
    has_reddit = merged["avg_sentiment"].notna()

    ax2 = ax.twinx()
    ax.plot(merged["x"], merged["close"], color=colors[ticker], linewidth=1.8, label="Close Price")
    bar_colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in merged.loc[has_reddit, "avg_sentiment"]]
    ax2.bar(merged.loc[has_reddit, "x"], merged.loc[has_reddit, "avg_sentiment"],
            color=bar_colors, alpha=0.5, width=0.6, label="Reddit Sentiment")
    ax2.axhline(0, color="grey", linewidth=0.6, linestyle="--")

    tick_idx    = merged["x"][::4].values
    tick_labels = merged["year_week"][::4].values
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(tick_labels, rotation=35, fontsize=8)
    ax.set_ylabel("Close Price (USD)", color=colors[ticker], fontsize=10)
    ax2.set_ylabel("Reddit Sentiment Score", fontsize=10)
    ax.set_title(f"{ticker_names[ticker]} ({ticker}) — Stock Price vs Reddit Sentiment",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax2.legend(loc="upper right", fontsize=9)

plt.suptitle("Weekly Stock Price vs Reddit Sentiment (Mar 2025 – Mar 2026)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\timeseries_overlay.png",
            bbox_inches="tight")
plt.show()

## 8. High Reddit Activity Weeks — Impact on Stock

Weeks where Reddit activity was unusually high (`high_reddit_activity == True`).
Did those weeks coincide with significant price movements?

In [ ]:
high_activity = reddit_weeks[reddit_weeks["high_reddit_activity"] == True][[
    "ticker", "year_week", "post_count", "avg_score", "avg_sentiment",
    "weekly_price_change_pct", "next_week_price_change_pct", "weekly_volatility"
]].sort_values("avg_score", ascending=False)

print(f"High Reddit activity weeks: {len(high_activity)}")
high_activity.reset_index(drop=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes,
    ["weekly_price_change_pct", "weekly_volatility"],
    ["Weekly Price Change (%)", "Weekly Volatility"]):

    data_high   = reddit_weeks[reddit_weeks["high_reddit_activity"] == True][col].dropna()
    data_normal = reddit_weeks[reddit_weeks["high_reddit_activity"] == False][col].dropna()

    ax.boxplot([data_normal, data_high],
               labels=["Normal Reddit\nWeeks", "High Reddit\nActivity Weeks"],
               patch_artist=True,
               boxprops=dict(facecolor="#AED6F1"),
               medianprops=dict(color="navy", linewidth=2))
    ax.set_title(f"{label}\nHigh vs Normal Reddit Activity", fontsize=11, fontweight="bold")
    ax.set_ylabel(label)

    t_stat, p_val = stats.ttest_ind(data_high, data_normal, equal_var=False)
    sig = "significant" if p_val < 0.05 else "not significant"
    ax.set_xlabel(f"Welch t-test  p = {p_val:.3f}  ({sig})", fontsize=9)

plt.tight_layout()
plt.savefig(r"C:\Users\Percival Mahwaya\Desktop\team-project-semester4\social-media-trends\high_activity_boxplot.png",
            bbox_inches="tight")
plt.show()

## 9. Summary of Findings

In [ ]:
summary = """
CORRELATION ANALYSIS SUMMARY
==============================
Dataset : 106 weeks with Reddit data (out of 168 total stock weeks)
Companies: AAPL, TSLA, AMZN  |  Period: Mar 2025 – Mar 2026

KEY FINDINGS
------------
1. SENTIMENT vs PRICE CHANGE
   - Weak to moderate correlation overall.
   - Reddit sentiment does NOT strongly predict same-week price change.
   - Slightly stronger next-week signal for TSLA — worth monitoring.
   - Suggests Reddit reacts to price events more than it predicts them.

2. REDDIT SCORE vs VOLATILITY
   - Higher Reddit engagement (score, comments) weakly correlates with higher volatility.
   - Most pronounced for AMZN: 2025-W17 score 43,237 / sentiment -0.40 / price +12.95%.
   - That spike was a reaction to a price event, not a predictor.

3. HIGH REDDIT ACTIVITY WEEKS
   - Weeks with high Reddit activity show slightly higher volatility.
   - Price change difference is not statistically significant (p > 0.05).

4. PER-COMPANY DIFFERENCES
   - TSLA : Strongest Reddit-stock relationship (most volatile, most discussed).
   - AMZN : One extreme outlier week dominates correlation signals.
   - AAPL : Weakest Reddit engagement relative to stock movement.

CONCLUSION
----------
Reddit sentiment is a lagging indicator — it reflects market events rather than
predicting them. However, spikes in Reddit engagement coincide with elevated
volatility, suggesting social media amplifies rather than causes price movement.
"""
print(summary)